# Fine-tune the ScholarCrew writer with LoRA

Trains a small adapter on top of an open model, using the checked examples built by `scholarcrew.finetune.build_dataset`.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells from top to bottom.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the code and the libraries
!git clone -q https://github.com/omsawant-ds/scholarcrew.git
%cd scholarcrew
!pip install -q -U transformers peft accelerate bitsandbytes

In [ ]:
# 3. Upload your training examples: choose data/sft/train.jsonl from your computer.
#    The file stays in this session. It is not on GitHub.
import os
from google.colab import files

os.makedirs('data/sft', exist_ok=True)
uploaded = files.upload()
name = next(iter(uploaded))
os.replace(name, 'data/sft/train.jsonl')
!wc -l data/sft/train.jsonl

In [ ]:
# 4. Train. About 15 to 40 minutes on a free T4 for 8B with ~100 short examples.
#    If the GPU runs out of memory, change the model to Qwen/Qwen3-4B or add --max-len 3072.
!PYTHONPATH=src python -m scholarcrew.finetune.train_lora --model Qwen/Qwen3-8B --out outputs/writer-lora

In [ ]:
# 5. Download the adapter (a few hundred MB), so it survives the session ending.
!zip -qr writer-lora.zip outputs/writer-lora -x 'outputs/writer-lora/checkpoints/*'
from google.colab import files
files.download('writer-lora.zip')

In [ ]:
# 6. Optional: also keep a private copy on Hugging Face (free).
#    Create a token at huggingface.co/settings/tokens with 'write' access, then run:
# from huggingface_hub import notebook_login
# notebook_login()
# !PYTHONPATH=src python -m scholarcrew.finetune.train_lora --model Qwen/Qwen3-8B --out outputs/writer-lora --epochs 3 --push-to-hub YOUR-USERNAME/scholarcrew-writer-lora